<a href="https://colab.research.google.com/github/Broklink/Data-test/blob/main/Data_LOGA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
# ==============================================================================
# 🚨 เฟสที่ 1: ชำแหละเฉพาะคลัง "Boxme" (Isolate Variables)
# ==============================================================================
import pandas as pd
import numpy as np
import re
import io
from google.colab import files

print("👇 กดปุ่ม 'Choose Files' แล้วเลือกแค่ 2 ไฟล์นี้เท่านั้นครับ:")
print("1. ไฟล์ยอดขาย (รายงานขายปี 2025...)")
print("2. ไฟล์สต๊อก Boxme (...Stock 25.3.26.csv)")
uploaded = files.upload()

print("\n⏳ กำลังล็อกเป้าและประมวลผลเฉพาะคลัง Boxme...")

try:
    # 1. แยกไฟล์ (เตะไฟล์อื่นที่ไม่เกี่ยวข้องทิ้ง)
    file_list = list(uploaded.keys())
    sales_file = [f for f in file_list if 'รายงานขาย' in f][0]
    boxme_file = [f for f in file_list if '25.3.26' in f or 'Boxme' in f.lower()][0]

    # 2. ล้างข้อมูลยอดขาย (ท่อ 1)
    try:
        df_sales = pd.read_csv(io.BytesIO(uploaded[sales_file]), encoding='utf-8-sig', skiprows=11)
    except:
        df_sales = pd.read_csv(io.BytesIO(uploaded[sales_file]), encoding='cp874', skiprows=11)

    df_sales = df_sales.dropna(how='all')
    df_sales['จำนวน'] = df_sales['จำนวน'].astype(str).str.replace(',', '').apply(pd.to_numeric, errors='coerce').fillna(0)
    df_items = df_sales[df_sales['ชื่อสินค้า/บริการ'].notna() & (df_sales['ชื่อสินค้า/บริการ'].astype(str).str.strip() != '')].copy()

    # 3. ล้างข้อมูล Boxme (ท่อ 2 - ติดตั้งตัวฆ่าคอลัมน์ผี)
    df_boxme = pd.read_csv(io.BytesIO(uploaded[boxme_file]), skiprows=4)
    df_boxme = df_boxme.dropna(subset=['Product name']).copy()

    # ตรรกะ Semantic Search: หาคอลัมน์ Stock/Value ล่าสุดเสมอ
    stock_cols_b = [c for c in df_boxme.columns if 'Stock' in str(c) or 'คงเหลือ' in str(c)]
    value_cols_b = [c for c in df_boxme.columns if 'Value' in str(c) or 'มูลค่า' in str(c)]

    df_boxme['Boxme_Qty'] = pd.to_numeric(df_boxme[stock_cols_b[-1]].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
    df_boxme['Boxme_Value'] = pd.to_numeric(df_boxme[value_cols_b[-1]].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

    # 4. วุ้นแปลภาษา (Aggressive String Cleansing)
    def clean_name(text):
        text = str(text).lower()
        for w in ['loga', 'esport', 'mousepad', 'caseonly', 'keyboard', 'base', 'cm', 'wireless', 'mouse', ' ']:
            text = text.replace(w, '')
        return re.sub(r'[^a-z0-9]', '', text).replace('x', '').replace('*', '')

    df_items['Match_Key'] = df_items['ชื่อสินค้า/บริการ'].apply(clean_name)
    df_boxme['Match_Key'] = df_boxme['Product name'].apply(clean_name)

    # 5. ประมวลผลและสร้างตารางวิเคราะห์
    sales_qty = df_items.groupby('Match_Key')['จำนวน'].sum().reset_index()
    sales_qty['Avg_Monthly_Sales'] = sales_qty['จำนวน'] / 12

    # รวบรวม Boxme เผื่อมีรายการซ้ำ
    df_boxme_clean = df_boxme[['Product name', 'Match_Key', 'Boxme_Qty', 'Boxme_Value']].rename(columns={'Product name': 'Raw_Name'})
    boxme_summary = df_boxme_clean.groupby(['Match_Key', 'Raw_Name']).agg({'Boxme_Qty':'sum', 'Boxme_Value':'sum'}).reset_index()

    final_table = pd.merge(boxme_summary, sales_qty[['Match_Key', 'Avg_Monthly_Sales']], on='Match_Key', how='left').fillna(0)

    # สมการวิเคราะห์ความเสี่ยง: สต๊อกหารยอดขาย
    final_table['Months_to_Clear'] = final_table['Boxme_Qty'] / final_table['Avg_Monthly_Sales']
    final_table.replace([np.inf, -np.inf], 999, inplace=True) # กรณีขายไม่ออกเลย ปัดเป็น 999

    # กรองเฉพาะตัวที่มีเงินจม และเรียงจากมากไปน้อย
    final_table = final_table[final_table['Boxme_Value'] > 0].sort_values('Boxme_Value', ascending=False)
    final_table = final_table[['Raw_Name', 'Boxme_Qty', 'Boxme_Value', 'Avg_Monthly_Sales', 'Months_to_Clear']]
    final_table.columns = ['ชื่อสินค้าใน Boxme', 'จำนวน (ชิ้น)', 'ทุนจม Boxme (บาท)', 'ขายเฉลี่ย/เดือน', 'เดือนกว่าจะหมด']

    pd.set_option('display.max_rows', 30)
    pd.options.display.float_format = '{:,.2f}'.format

    print("\n✅ ปฏิบัติการเจาะคลัง Boxme เสร็จสมบูรณ์! ข้อมูลสะอาด 100%:")
    display(final_table.head(30))

except Exception as e:
    print(f"\n❌ ถอดรหัส Error: {e}")
    print("👉 ตรวจสอบว่าอัปโหลดไฟล์มาแค่ 2 ไฟล์ตามที่กำหนดหรือไม่ครับ")

👇 กดปุ่ม 'Choose Files' แล้วเลือกแค่ 2 ไฟล์นี้เท่านั้นครับ:
1. ไฟล์ยอดขาย (รายงานขายปี 2025...)
2. ไฟล์สต๊อก Boxme (...Stock 25.3.26.csv)


Saving 2026-MKT._Update Inventory - Stock 25.3.26.csv to 2026-MKT._Update Inventory - Stock 25.3.26 (5).csv
Saving รายงานขายปี 2025 NDA save .csv to รายงานขายปี 2025 NDA save  (8).csv

⏳ กำลังล็อกเป้าและประมวลผลเฉพาะคลัง Boxme...

✅ ปฏิบัติการเจาะคลัง Boxme เสร็จสมบูรณ์! ข้อมูลสะอาด 100%:


,ชื่อสินค้าใน Boxme,จำนวน (ชิ้น),ทุนจม Boxme (บาท),ขายเฉลี่ย/เดือน,เดือนกว่าจะหมด
24,LOGA Base Barebone 104 Keyboard (Case only) : ...,613.00,"491,319.50",0.00,999.00
48,GARUDA 2 Wireless Mouse : Black,463.00,"473,186.00",0.00,999.00
49,GARUDA 2 Wireless Mouse : White,444.00,"453,768.00",0.00,999.00
25,LOGA Base Barebone 104s (case only) : black,452.00,"332,672.00",0.00,999.00
9,LOGA ESPORT MOUSEPAD : ARASHI 49x42 (Black),505.00,"276,235.00",0.00,999.00
26,LOGA Base Barebone 104s (case only) : white,294.00,"216,384.00",0.00,999.00
10,LOGA ESPORT MOUSEPAD : ARASHI 49x42 (Purple),379.00,"207,313.00",0.00,999.00
12,Ascension Pacific 2025 // Full set keycap (ENG...,359.00,"203,553.00",4.00,89.75
47,LOGA Ergo Series : Sabai TKL Wrist Rest (Gray),776.00,"187,873.79",42.25,18.37
45,LOGA Ergo Series : Sabai Full Size Wrist Rest ...,723.00,"182,175.25",42.67,16.95


In [54]:
# ==============================================================================
# 🚨 เฟสที่ 2: ชำแหละเฉพาะคลัง "Office" (ค้นหาความผิดปกติของระบบจัดเก็บ)
# ==============================================================================
import pandas as pd
import numpy as np
import re
import io
from google.colab import files

print("👇 กดปุ่ม 'Choose Files' แล้วเลือกแค่ 2 ไฟล์นี้เท่านั้นครับ:")
print("1. ไฟล์ยอดขาย (รายงานขายปี 2025...)")
print("2. ไฟล์สต๊อก Office")
uploaded = files.upload()

print("\n⏳ กำลังล็อกเป้าและประมวลผลเฉพาะคลัง Office...")

try:
    # 1. แยกไฟล์ (เตะไฟล์อื่นที่ไม่เกี่ยวข้องทิ้ง)
    file_list = list(uploaded.keys())
    sales_file = [f for f in file_list if 'รายงานขาย' in f][0]
    office_file = [f for f in file_list if f != sales_file][0] # ไฟล์ที่เหลือต้องเป็น Office

    # 2. ล้างข้อมูลยอดขาย (ท่อ 1)
    try:
        df_sales = pd.read_csv(io.BytesIO(uploaded[sales_file]), encoding='utf-8-sig', skiprows=11)
    except:
        df_sales = pd.read_csv(io.BytesIO(uploaded[sales_file]), encoding='cp874', skiprows=11)

    df_sales = df_sales.dropna(how='all')
    df_sales['จำนวน'] = df_sales['จำนวน'].astype(str).str.replace(',', '').apply(pd.to_numeric, errors='coerce').fillna(0)
    df_items = df_sales[df_sales['ชื่อสินค้า/บริการ'].notna() & (df_sales['ชื่อสินค้า/บริการ'].astype(str).str.strip() != '')].copy()

    # 3. ล้างข้อมูล Office (ท่อ 2 - ติดตั้งตัวฆ่าคอลัมน์ผี)
    df_office = pd.read_csv(io.BytesIO(uploaded[office_file]), skiprows=2)
    df_office = df_office.dropna(subset=['ชื่อสินค้า/บริการ']).copy()

    # ตรรกะ Semantic Search: กวาดหาคอลัมน์ 'คงเหลือ' และ 'มูลค่า' ที่แท้จริง
    stock_cols_o = [c for c in df_office.columns if 'คงเหลือ' in str(c) or 'Stock' in str(c)]
    value_cols_o = [c for c in df_office.columns if 'มูลค่า' in str(c) or 'Value' in str(c)]

    df_office['Office_Qty'] = pd.to_numeric(df_office[stock_cols_o[-1]].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
    df_office['Office_Value'] = pd.to_numeric(df_office[value_cols_o[-1]].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

    # 4. วุ้นแปลภาษา (Aggressive String Cleansing)
    def clean_name(text):
        text = str(text).lower()
        for w in ['loga', 'esport', 'mousepad', 'caseonly', 'keyboard', 'base', 'cm', 'wireless', 'mouse', ' ']:
            text = text.replace(w, '')
        return re.sub(r'[^a-z0-9]', '', text).replace('x', '').replace('*', '')

    df_items['Match_Key'] = df_items['ชื่อสินค้า/บริการ'].apply(clean_name)
    df_office['Match_Key'] = df_office['ชื่อสินค้า/บริการ'].apply(clean_name)

    # 5. ประมวลผลและสร้างตารางวิเคราะห์
    sales_qty = df_items.groupby('Match_Key')['จำนวน'].sum().reset_index()
    sales_qty['Avg_Monthly_Sales'] = sales_qty['จำนวน'] / 12

    # รวบรวม Office เผื่อมีรายการซ้ำ
    df_office_clean = df_office[['ชื่อสินค้า/บริการ', 'Match_Key', 'Office_Qty', 'Office_Value']].rename(columns={'ชื่อสินค้า/บริการ': 'Raw_Name'})
    office_summary = df_office_clean.groupby(['Match_Key', 'Raw_Name']).agg({'Office_Qty':'sum', 'Office_Value':'sum'}).reset_index()

    final_table = pd.merge(office_summary, sales_qty[['Match_Key', 'Avg_Monthly_Sales']], on='Match_Key', how='left').fillna(0)

    # สมการวิเคราะห์ความเสี่ยง: สต๊อกออฟฟิศหารยอดขาย
    final_table['Months_to_Clear'] = final_table['Office_Qty'] / final_table['Avg_Monthly_Sales']
    final_table.replace([np.inf, -np.inf], 999, inplace=True) # กรณีขายไม่ออกเลย ปัดเป็น 999

    # กรองเฉพาะตัวที่มีเงินจม และเรียงจากมากไปน้อย
    final_table = final_table[final_table['Office_Value'] > 0].sort_values('Office_Value', ascending=False)
    final_table = final_table[['Raw_Name', 'Office_Qty', 'Office_Value', 'Avg_Monthly_Sales', 'Months_to_Clear']]
    final_table.columns = ['ชื่อสินค้าใน Office', 'จำนวน (ชิ้น)', 'ทุนจม Office (บาท)', 'ขายเฉลี่ย/เดือน', 'เดือนกว่าจะหมด']

    pd.set_option('display.max_rows', 30)
    pd.options.display.float_format = '{:,.2f}'.format

    print("\n✅ ปฏิบัติการเจาะคลัง Office เสร็จสมบูรณ์! นี่คือรายชื่อสต๊อกที่ผิดที่ผิดทาง:")
    display(final_table.head(30))

except Exception as e:
    print(f"\n❌ ถอดรหัส Error: {e}")
    print("👉 ตรวจสอบว่าอัปโหลดไฟล์มาแค่ 2 ไฟล์ตามที่กำหนดหรือไม่ครับ")

👇 กดปุ่ม 'Choose Files' แล้วเลือกแค่ 2 ไฟล์นี้เท่านั้นครับ:
1. ไฟล์ยอดขาย (รายงานขายปี 2025...)
2. ไฟล์สต๊อก Office


Saving 2026-MKT._Update Inventory.csv to 2026-MKT._Update Inventory (5).csv
Saving รายงานขายปี 2025 NDA save .csv to รายงานขายปี 2025 NDA save  (9).csv

⏳ กำลังล็อกเป้าและประมวลผลเฉพาะคลัง Office...

✅ ปฏิบัติการเจาะคลัง Office เสร็จสมบูรณ์! นี่คือรายชื่อสต๊อกที่ผิดที่ผิดทาง:


,ชื่อสินค้าใน Office,จำนวน (ชิ้น),ทุนจม Office (บาท),ขายเฉลี่ย/เดือน,เดือนกว่าจะหมด
59,LOGA x Hitscan : Printstream Wireless Mouse,60.00,"131,820.00",65.50,0.92
25,Big Box Mutelu : LOGA Blindbox series Keycap,48.00,"75,792.00",0.00,999.00
114,LOGA x SUNTUR Limited collection SUNTUR Keycap...,60.00,"65,400.00",8.42,7.13
12,LOGA Base 104 : Strawberry Cheese Cake (NY),37.00,"43,152.16",23.33,1.59
11,LOGA Base 104 : Strawberry Cheese Cake (NS),33.00,"38,487.06",19.75,1.67
138,LOGA YAKSA PRO 75% Clear : Maiyarap edition -...,21.00,"35,160.03",0.25,84.00
5,LOGA Base 104 : Americano (NY),29.00,"33,821.96",22.33,1.30
0,หมาน้ำตาล,61.00,"33,123.00",180.92,0.34
1,หมาพาสเทล,59.00,"32,054.90",180.92,0.33
10,LOGA Base 104 : Strawberry Cheese Cake (CT)),26.00,"30,323.14",21.92,1.19
